# CSCI 3151 â€” Foundations of Machine Learning

## M31 â€” Backpropagation & automatic differentiation: PyTorch autograd in practice

In this mini-lab you will connect the **math of gradients and backpropagation** from the M31 video to
the **actual tensors and `.grad` fields** you see when training small neural networks in PyTorch.
We will:

- Start with a tiny scalar function and see how PyTorch applies the chain rule automatically.
- Build and inspect a small multi-layer perceptron (MLP) for a toy 2D classification task.
- Check that PyTorch's gradients match finite-difference approximations.

Along the way, you will practice reading computation graphs from code and reasoning about
how gradients flow through parameters during the backward pass.

**This mini-lab reinforcesâ€¦**

- The chain rule and backpropagation as *local* computations on a computational graph.
- How PyTorch builds a dynamic graph on the forward pass and fills in `.grad` on `backward()`.
- How to inspect and sanity-check gradients in small networks.
- How finite-difference checks can help debug suspicious training behaviour.


## Assumptions & scope

- You are comfortable with basic Python, NumPy, and writing simple functions.
- You have seen PyTorch tensors before (e.g., in M30) and know how to run code cells in Jupyter.
- We work with **small synthetic datasets** so that you can focus on gradients, not on data cleaning.
- We only scratch the surface of forward-mode vs reverse-mode autodiff; the emphasis here is on
  **reverse-mode (backprop)** and how it appears in PyTorch.

Time is short, so we will not:

- Derive full matrix-calculus expressions for every layer in the MLP.
- Explore advanced PyTorch features like custom `autograd.Function` or distributed training.
- Worry about perfect generalization: the goal is to understand and **trust but verify** gradients
  on a small example.


In [ ]:
# === Setup: imports and reproducibility ======================================

import numpy as np
import pandas as pd  # not heavily used, but handy if you want small tables
import matplotlib.pyplot as plt

import torch
from torch import nn

from sklearn.datasets import make_moons

# For reasonably pretty plots in notebooks
%matplotlib inline

SEED = 3151
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
# === Tiny 2D classification dataset (two moons) ==============================

# We'll use the same two-moons toy dataset that has appeared earlier in the course.

X_np, y_np = make_moons(n_samples=300, noise=0.2, random_state=SEED)

X = torch.from_numpy(X_np).float().to(device)
y = torch.from_numpy(y_np).float().unsqueeze(1).to(device)  # shape (N, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

plt.figure(figsize=(4, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=15)
plt.title("Two-moons toy data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.tight_layout()
plt.show()


## Core Task 1 â€” Warm-up: autograd on a scalar function

We'll start with a **single scalar parameter** and a simple scalar-valued function:

\begin{align}
f(w) = (w^2 + 3w + 1)^2.
\end{align}

1. Create a 1D tensor `w` with `requires_grad=True`.
2. Compute `f = (w**2 + 3*w + 1)**2` using PyTorch operations.
3. Call `f.backward()` to ask PyTorch for $\partial f / \partial w$.
4. Compare `w.grad` to the **analytic derivative** at that point.

> **Reminder:** The analytic derivative is
> \( f'(w) = 2 (w^2 + 3w + 1) (2w + 3). \)

Try a few different values of `w` and check that the autograd and analytic gradients match (up to small
floating-point differences).


In [ ]:
# === Core Task 1: scalar autograd ===========================================

def analytic_df_dw(w_value: float) -> float:
    """Analytic derivative f'(w) for f(w) = (w^2 + 3w + 1)^2, evaluated at a Python float."""
    return 2.0 * (w_value**2 + 3.0 * w_value + 1.0) * (2.0 * w_value + 3.0)

# TODO: pick a few different starting values for w here
for w_start in [-2.0, -0.5, 1.0, 3.0]:
    # 1. Create a tensor with gradient tracking
    w = torch.tensor(w_start, dtype=torch.float32, requires_grad=True, device=device)

    # 2. Forward pass: compute f(w)
    # TODO: replace ... with the expression for f(w)
    f = ...

    # 3. Backward pass: compute df/dw
    # (Make sure gradient is cleared first if you're reusing w; here we recreate w each time.)
    f.backward()

    auto_grad = w.grad.item()
    true_grad = analytic_df_dw(w_start)

    print(f"w = {w_start:5.2f} | autograd df/dw = {auto_grad:+.6f} | analytic = {true_grad:+.6f}")


## Core Task 2 â€” A tiny MLP and its gradients

Next, let's build a small **two-layer MLP** for the two-moons data and inspect the gradients flowing
through its parameters.

We'll use the architecture:

- Input: 2D features `(x1, x2)`
- Hidden layer: linear `Linear(2 â†’ 8)` + `tanh` nonlinearity
- Output layer: linear `Linear(8 â†’ 1)` producing a single logit for binary classification

We'll train with **binary cross-entropy with logits** (`BCEWithLogitsLoss`).

Your tasks:

1. Complete the `TinyMLP` class below.
2. Run one forward and backward pass on the entire dataset.
3. Print the **gradient shapes and norms** for each parameter tensor.

Think about how this relates to the computational-graph picture in the M31 video: each gradient is
the result of many local chain-rule multiplications.


In [ ]:
# === Core Task 2: define a tiny MLP =========================================

class TinyMLP(nn.Module):
    def __init__(self, in_dim: int = 2, hidden_dim: int = 8, out_dim: int = 1):
        super().__init__()

        # TODO: define the two linear layers
        # Hint: use nn.Linear(in_dim, hidden_dim) and nn.Linear(hidden_dim, out_dim)
        ...

        # We'll use tanh as our nonlinearity (smooth and has a nice derivative).
        self.activation = torch.tanh

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement the forward pass: x -> fc1 -> tanh -> fc2
        h = ...
        h = ...
        out = ...
        return out


model = TinyMLP().to(device)
loss_fn = nn.BCEWithLogitsLoss()

model


In [ ]:
# One forward + backward pass on the full dataset

# IMPORTANT: always zero gradients before calling backward, otherwise they accumulate.
model.zero_grad()

logits = model(X)           # shape (N, 1)
loss = loss_fn(logits, y)   # scalar

print("Loss value (single batch):", loss.item())

loss.backward()             # backprop from scalar loss to all parameters

print("\nParameter gradient norms:")
for name, param in model.named_parameters():
    if param.grad is None:
        print(f"- {name:10s}: grad is None (did this parameter participate in the computation?)")
    else:
        grad_norm = param.grad.norm().item()
        print(f"- {name:10s}: shape = {tuple(param.grad.shape)}, ||grad||â‚‚ = {grad_norm:.4f}")


## Core Task 3 â€” Gradient checking with finite differences

Autograd systems are complex; it is useful to occasionally **check** their answers using
simple finite-difference approximations.

For a single scalar parameter $\theta$, we can approximate the derivative of the loss $L(\theta)$ via
a **central difference**:

\begin{align}
\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}.
\end{align}

Here you'll:

1. Pick **one weight** from the first linear layer (e.g., `fc1.weight[0, 0]`).
2. Write a function that perturbs just that entry by `+eps` and `-eps`, recomputes the loss,
   and returns the finite-difference estimate.
3. Compare this numerical gradient to the corresponding entry in `param.grad` from autograd.

Use a small `eps` like `1e-4` or `1e-5`. The two values should be close but not identical (roundoff error).
If they differ wildly, something is wrong with either the gradient or the checking code.


In [ ]:
# === Core Task 3: finite-difference gradient check ==========================

def loss_on_full_data(model: nn.Module) -> torch.Tensor:
    """Compute BCE loss on the (fixed) full two-moons dataset."""
    logits = model(X)
    return loss_fn(logits, y)


def finite_difference_grad(model: nn.Module, param: torch.nn.Parameter, index_tuple, eps: float = 1e-4) -> float:
    """Approximate dL/dtheta for a single parameter entry using central differences.

    Args:
        model: the TinyMLP instance (will be modified in-place but restored).
        param: a parameter tensor, e.g., model.fc1.weight.
        index_tuple: index selecting a single scalar entry in param, e.g., (0, 0).
        eps: small perturbation size.
    """
    # Make sure we don't accidentally track these perturbations in the autograd graph.
    with torch.no_grad():
        original_value = param[index_tuple].item()

        # L(theta + eps)
        param[index_tuple] = original_value + eps
        loss_plus = loss_on_full_data(model).item()

        # L(theta - eps)
        param[index_tuple] = original_value - eps
        loss_minus = loss_on_full_data(model).item()

        # Restore original value
        param[index_tuple] = original_value

    return (loss_plus - loss_minus) / (2.0 * eps)


# Pick a single scalar parameter to inspect (you can change these indices)
param = model.fc1.weight  # shape (hidden_dim, in_dim)
idx = (0, 0)

# 1. Get autograd's answer (make sure grads are fresh!)
model.zero_grad()
loss = loss_on_full_data(model)
loss.backward()

auto_grad_value = param.grad[idx].item()

# 2. Get finite-difference estimate
fd_grad_value = finite_difference_grad(model, param, idx, eps=1e-4)

print(f"Parameter: fc1.weight{idx}")
print(f"Autograd gradient      â‰ˆ {auto_grad_value:+.6f}")
print(f"Finite-diff gradient   â‰ˆ {fd_grad_value:+.6f}")
print(f"Absolute difference    = {abs(auto_grad_value - fd_grad_value):.6e}")


## Stretch tasks (if you have extra time)

If you finish the core tasks, try one or more of the following:

1. **Mini training loop:**
   - Write a loop that runs for, say, 200 epochs of full-batch gradient descent on the two-moons data.
   - Track the loss over epochs in a Python list and plot it.
   - Experiment with different learning rates to see when training is stable vs unstable.

2. **Gradient accumulation gotcha:**
   - Modify your training loop to *forget* the `model.zero_grad()` / `optimizer.zero_grad()` call.
   - Watch what happens to the gradient norms and loss values.
   - Explain, in words, why gradients accumulate by default in PyTorch and how this relates to backprop.

3. **Inspecting intermediate tensors:**
   - In the `forward` method of `TinyMLP`, print or log the mean / std of the hidden activations `h`.
   - How do these change during training? How might that connect to vanishing/exploding gradients in
     deeper networks (a topic for a later module)?

Use markdown cells below your code to briefly summarise what you observed.
